# Data cleaning pipeline

This notebook builds MatchCast's processed match table from the raw
`martj42/international_results` snapshot. It is built up section by section,
one cleaning concern at a time, matching the schema defined in
`01_processed_match_schema.ipynb`. Each section documents its rule and
validates it before the next section runs.

In [1]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path("../data/raw/international_results.csv")
PROCESSED_PATH = Path("../data/processed/matches.csv")

raw = pd.read_csv(RAW_PATH, dtype="string", keep_default_na=False)
raw["source_row_number"] = range(len(raw))

print(f"Loaded {len(raw)} raw rows from {RAW_PATH}")

Loaded 49501 raw rows from ..\data\raw\international_results.csv


## Parse and sort match dates

Dates are parsed with an explicit `YYYY-MM-DD` format so malformed values
surface as `NaT` instead of being silently misparsed. Rows are then sorted by
`date` using a stable sort with `source_row_number` as the deterministic
tiebreaker, so matches played on the same day keep their original upstream
order every time this notebook runs.

In [2]:
matches = raw.copy()
matches["date"] = pd.to_datetime(
    matches["date"].str.strip(), format="%Y-%m-%d", errors="raise"
)
matches = matches.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)

print(f"Date range: {matches['date'].min().date()} to {matches['date'].max().date()}")

Date range: 1872-11-30 to 2026-07-06


### Validation: date parsing and sort determinism

In [3]:
assert matches["date"].notna().all(), "every row must have a parsed date"
assert len(matches) == len(raw), "sorting must not drop or add rows"
assert matches["date"].is_monotonic_increasing, "rows must be sorted by date"

resorted = matches.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)
assert matches["source_row_number"].tolist() == resorted["source_row_number"].tolist(), (
    "re-sorting the already-sorted table must be a no-op (deterministic order)"
)

same_day = matches[matches.duplicated(subset="date", keep=False)]
for _, group in same_day.groupby("date"):
    assert group["source_row_number"].is_monotonic_increasing, (
        "same-day matches must preserve original upstream order"
    )

print("Date parsing and deterministic sort validated.")

Date parsing and deterministic sort validated.


## Normalize team names and preserve originals

Team names are normalized (surrounding whitespace stripped, internal
whitespace collapsed) and passed through an explicit alias mapping before use
in any downstream feature or model. The unnormalized values are preserved in
`original_home_team` / `original_away_team` so every processed row remains
traceable to the exact upstream string, even if a future refresh or a
supplementary source (see `docs/data-sources.md`) introduces a name this
mapping does not yet cover.

The current snapshot's team names are already whitespace-clean and
single-form (verified below), so `TEAM_NAME_ALIASES` starts empty. The
mapping exists so a future snapshot revealing spelling variants (for example
an alternate name for the same team) can be corrected in one place without
rewriting the pipeline.

In [4]:
TEAM_NAME_ALIASES: dict[str, str] = {}


def normalize_team_name(name: str, aliases: dict[str, str] = TEAM_NAME_ALIASES) -> str:
    """Collapse whitespace and apply the explicit alias mapping."""
    normalized = " ".join(name.strip().split())
    return aliases.get(normalized, normalized)


matches["original_home_team"] = matches["home_team"]
matches["original_away_team"] = matches["away_team"]
matches["home_team"] = matches["home_team"].map(normalize_team_name)
matches["away_team"] = matches["away_team"].map(normalize_team_name)

print(f"Distinct normalized teams: {pd.unique(matches[['home_team', 'away_team']].values.ravel()).size}")

Distinct normalized teams: 336


### Validation: normalization and traceability

In [5]:
for column in ("home_team", "away_team"):
    assert (matches[column] == matches[column].str.strip()).all(), (
        f"{column} must not have leading/trailing whitespace"
    )
    assert (~matches[column].str.contains(r"\s{2,}", regex=True)).all(), (
        f"{column} must not contain repeated internal whitespace"
    )
    assert (matches[column] != "").all(), f"{column} must not be empty after normalization"

# Normalization must be idempotent.
assert matches["home_team"].map(normalize_team_name).equals(matches["home_team"])
assert matches["away_team"].map(normalize_team_name).equals(matches["away_team"])

# Original values must be preserved untouched for traceability.
assert matches["original_home_team"].equals(raw.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)["home_team"])
assert matches["original_away_team"].equals(raw.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)["away_team"])

print("Team name normalization and traceability validated.")

Team name normalization and traceability validated.


## Handle missing, duplicate, malformed, and impossible records

Documented rules applied in order:

1. **Missing scores.** Rows with `NA` for both scores are scheduled fixtures
   that have not been played yet (see `docs/data-sources.md`). They carry no
   outcome to learn from, so they are excluded from the processed table and
   the excluded count is reported. A row with only one score missing would be
   a malformed partial record; none exist in this snapshot, but the rule is
   the same: exclude it, since it cannot support a `result`.
2. **Duplicates.** Exact duplicate rows (identical on every raw column) are
   removed, keeping the first occurrence by `source_row_number` so the kept
   copy is deterministic.
3. **Malformed dates.** Already handled: date parsing above uses
   `errors="raise"`, so a malformed date fails the pipeline instead of being
   silently dropped or coerced.
4. **Malformed `neutral` flag.** Must be exactly `TRUE` or `FALSE`; anything
   else fails the pipeline rather than being guessed.
5. **Impossible scores.** Negative or non-integer scores are rejected.

In [6]:
before_dedupe = len(matches)
matches = matches.drop_duplicates(
    subset=[c for c in matches.columns if c not in ("source_row_number",)],
    keep="first",
)
duplicate_count = before_dedupe - len(matches)

missing_score_mask = (matches["home_score"] == "NA") | (matches["away_score"] == "NA")
partial_score_mask = (matches["home_score"] == "NA") ^ (matches["away_score"] == "NA")
assert not partial_score_mask.any(), "a row must have both scores or neither"

scheduled_fixture_count = int(missing_score_mask.sum())
matches = matches.loc[~missing_score_mask].copy()

assert matches["neutral"].isin({"TRUE", "FALSE"}).all(), "neutral must be TRUE or FALSE"
matches["neutral"] = matches["neutral"].map({"TRUE": True, "FALSE": False}).astype("boolean")

matches["home_score"] = pd.to_numeric(matches["home_score"], errors="raise").astype("Int64")
matches["away_score"] = pd.to_numeric(matches["away_score"], errors="raise").astype("Int64")
assert (matches["home_score"] >= 0).all() and (matches["away_score"] >= 0).all(), (
    "scores must be non-negative"
)

print(
    f"Removed {duplicate_count} exact duplicate rows and "
    f"{scheduled_fixture_count} scheduled fixtures without a final score. "
    f"{len(matches)} completed matches remain."
)

Removed 0 exact duplicate rows and 8 scheduled fixtures without a final score. 49493 completed matches remain.

### Validation: missing, duplicate, and malformed record handling

In [7]:
assert scheduled_fixture_count == 8, "expected 8 scheduled fixtures in this pinned snapshot"
assert (matches["home_score"].notna() & matches["away_score"].notna()).all(), (
    "every remaining row must have both final scores"
)
assert not matches.duplicated(
    subset=[c for c in matches.columns if c not in ("source_row_number",)]
).any(), "no exact duplicate rows may remain"
assert matches["neutral"].dtype == "boolean"
assert matches["neutral"].notna().all(), "neutral must never be null after conversion"
assert len(matches) == before_dedupe - duplicate_count - scheduled_fixture_count

print("Missing, duplicate, and malformed record handling validated.")

Missing, duplicate, and malformed record handling validated.


## Derive `result` and `goal_difference`

`result` is the match outcome from the home team's perspective (`H`/`D`/`A`),
and `goal_difference` is `home_score - away_score`. Both are derived purely
from already-known final scores, so they carry no information leakage risk.

In [8]:
matches["goal_difference"] = matches["home_score"] - matches["away_score"]
matches["result"] = pd.Series("D", index=matches.index, dtype="string")
matches.loc[matches["goal_difference"] > 0, "result"] = "H"
matches.loc[matches["goal_difference"] < 0, "result"] = "A"

print(matches["result"].value_counts())

result
H    24256
A    13981
D    11256
Name: count, dtype: Int64


### Validation: derived fields

In [9]:
assert set(matches["result"].unique()) <= {"H", "D", "A"}
assert (matches["goal_difference"] == matches["home_score"] - matches["away_score"]).all()
assert (matches.loc[matches["result"] == "H", "goal_difference"] > 0).all()
assert (matches.loc[matches["result"] == "A", "goal_difference"] < 0).all()
assert (matches.loc[matches["result"] == "D", "goal_difference"] == 0).all()
assert matches["result"].notna().all()
assert matches["goal_difference"].notna().all()

print("Derived result and goal_difference fields validated.")

Derived result and goal_difference fields validated.


## Assign `match_id` and save the processed table

`match_id` is derived from `source_row_number`, so it is stable across reruns
and traces every processed row back to its exact line in the pinned raw CSV.
Columns are ordered to match `PROCESSED_MATCH_COLUMNS` from
`01_processed_match_schema.ipynb`, and the table is written to
`data/processed/matches.csv` so the command is a pure, reproducible function
of the pinned raw snapshot.

In [10]:
matches["match_id"] = matches["source_row_number"].map(lambda n: f"R{n:05d}")

PROCESSED_MATCH_COLUMNS = (
    "match_id",
    "source_row_number",
    "date",
    "home_team",
    "away_team",
    "original_home_team",
    "original_away_team",
    "home_score",
    "away_score",
    "tournament",
    "city",
    "country",
    "neutral",
    "result",
    "goal_difference",
)
matches = matches[list(PROCESSED_MATCH_COLUMNS)]

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
matches.to_csv(PROCESSED_PATH, index=False)

print(f"Wrote {len(matches)} processed matches to {PROCESSED_PATH}")

Wrote 49493 processed matches to ..\data\processed\matches.csv


### Validation: schema conformance and reproducibility

In [11]:
assert list(matches.columns) == list(PROCESSED_MATCH_COLUMNS)
assert matches["match_id"].is_unique, "match_id must uniquely identify a row"
assert matches["match_id"].notna().all()

reloaded = pd.read_csv(PROCESSED_PATH, dtype="string", keep_default_na=False)
assert len(reloaded) == len(matches)
assert list(reloaded.columns) == list(PROCESSED_MATCH_COLUMNS)
assert reloaded["match_id"].tolist() == matches["match_id"].tolist(), (
    "row order on disk must match the in-memory deterministic order"
)

print("Processed table schema and reproducibility validated.")

Processed table schema and reproducibility validated.

## Tests: cleaning rules, derived fields, duplicates, and determinism

The validations above confirm the pipeline behaves correctly on the real
snapshot, but that snapshot happens to contain no duplicates, no malformed
values, and no messy team names. This section exercises the same cleaning
rules against a small, hand-built fixture that deliberately contains those
edge cases, so the rules are actually tested rather than only run on already-clean
data.

In [12]:
def clean_fixture(df: pd.DataFrame) -> pd.DataFrame:
    """Apply the notebook's cleaning rules to an arbitrary raw-shaped frame."""
    df = df.copy()
    df["source_row_number"] = range(len(df))
    df["date"] = pd.to_datetime(df["date"].str.strip(), format="%Y-%m-%d", errors="raise")
    df = df.sort_values(by=["date", "source_row_number"], kind="stable").reset_index(drop=True)

    df["original_home_team"] = df["home_team"]
    df["original_away_team"] = df["away_team"]
    df["home_team"] = df["home_team"].map(normalize_team_name)
    df["away_team"] = df["away_team"].map(normalize_team_name)

    df = df.drop_duplicates(
        subset=[c for c in df.columns if c != "source_row_number"], keep="first"
    )
    missing = (df["home_score"] == "NA") | (df["away_score"] == "NA")
    assert not ((df["home_score"] == "NA") ^ (df["away_score"] == "NA")).any()
    df = df.loc[~missing].copy()

    assert df["neutral"].isin({"TRUE", "FALSE"}).all()
    df["neutral"] = df["neutral"].map({"TRUE": True, "FALSE": False}).astype("boolean")
    df["home_score"] = pd.to_numeric(df["home_score"], errors="raise").astype("Int64")
    df["away_score"] = pd.to_numeric(df["away_score"], errors="raise").astype("Int64")
    assert (df["home_score"] >= 0).all() and (df["away_score"] >= 0).all()

    df["goal_difference"] = df["home_score"] - df["away_score"]
    df["result"] = pd.Series("D", index=df.index, dtype="string")
    df.loc[df["goal_difference"] > 0, "result"] = "H"
    df.loc[df["goal_difference"] < 0, "result"] = "A"
    return df

In [13]:
fixture = pd.DataFrame(
    {
        "date": [
            "1990-06-08",
            "1990-06-08",
            "1990-06-08",
            "1990-06-09",
            "1990-06-10",
        ],
        "home_team": ["  Brazil ", "Brazil", "Brazil", "Italy", "Spain"],
        "away_team": ["Sweden", "Sweden", "Sweden", "Austria", "Uruguay"],
        "home_score": ["2", "1", "1", "1", "NA"],
        "away_score": ["1", "0", "0", "1", "NA"],
        "tournament": ["Friendly"] * 5,
        "city": ["Rio"] * 5,
        "country": ["Brazil", "Brazil", "Brazil", "Italy", "Spain"],
        "neutral": ["FALSE", "TRUE", "TRUE", "FALSE", "FALSE"],
    },
    dtype="string",
)
# Row 0 is a whitespace variant of row 1's team name with a different score
# (not a duplicate); rows 1 and 2 are exact duplicates; row 4 is a scheduled
# fixture with no final score.
cleaned_fixture = clean_fixture(fixture)

In [14]:
# Duplicate handling: the exact duplicate (rows 1 and 2) collapses to one row.
assert len(cleaned_fixture) == 3, "expected 3 rows: whitespace variant, deduped pair, and Italy-Austria"

# Missing-score handling: the scheduled Spain-Uruguay fixture is excluded.
assert not (cleaned_fixture["home_team"] == "Spain").any()

# Whitespace normalization: leading/trailing space is stripped but the row is
# kept distinct because its score differs from the exact duplicate pair.
assert (cleaned_fixture["home_team"] == "Brazil").sum() == 2
assert cleaned_fixture["home_team"].str.strip().eq(cleaned_fixture["home_team"]).all()

# Derived fields: result and goal_difference match the fixture's known scores.
brazil_sweden = cleaned_fixture[cleaned_fixture["original_home_team"] == "Brazil"].iloc[0]
assert brazil_sweden["goal_difference"] == 1
assert brazil_sweden["result"] == "H"
italy_austria = cleaned_fixture[cleaned_fixture["home_team"] == "Italy"].iloc[0]
assert italy_austria["goal_difference"] == 0
assert italy_austria["result"] == "D"

# Deterministic output: cleaning the same fixture twice gives identical results.
repeat = clean_fixture(fixture)
pd.testing.assert_frame_equal(
    cleaned_fixture.reset_index(drop=True), repeat.reset_index(drop=True)
)

# Malformed input is rejected rather than silently coerced.
malformed_date = fixture.copy()
malformed_date.loc[0, "date"] = "08/06/1990"
try:
    clean_fixture(malformed_date)
    raise AssertionError("malformed date must raise")
except ValueError:
    pass

negative_score = fixture.copy()
negative_score.loc[0, "home_score"] = "-1"
negative_score_rejected = False
try:
    clean_fixture(negative_score)
except AssertionError:
    negative_score_rejected = True
assert negative_score_rejected, "negative score must raise"

print("Cleaning-rule, derived-field, duplicate, and determinism tests passed.")

Cleaning-rule, derived-field, duplicate, and determinism tests passed.
